# 🔍 Notebook 03 — Exploratory Data Analysis (EDA)
**Urban Taxi Demand Pattern Analysis**

This notebook explores the cleaned data through visualisations and summary statistics. Charts are saved to `outputs/eda/`.

---

In [ ]:
import sys, os
sys.path.append(os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from utils import DATA_PROCESSED, OUTPUT_EDA, save_figure, load_cleaned

# ── Style ────────────────────────────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='viridis')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

OUTPUT_EDA.mkdir(parents=True, exist_ok=True)

## 1. Load All Cleaned Data

In [ ]:
YEAR = 2024
MONTHS = list(range(1, 7))

dfs = []
for taxi_type in ['yellow', 'green']:
    for month in MONTHS:
        try:
            df = load_cleaned(taxi_type, YEAR, month)
            dfs.append(df)
            print(f"  ✅ Loaded cleaned_{taxi_type}_{YEAR}-{month:02d} — {len(df):,} rows")
        except FileNotFoundError:
            print(f"  ⏭️  Not found: cleaned_{taxi_type}_{YEAR}-{month:02d}")

df = pd.concat(dfs, ignore_index=True)
print(f"\n📊 Combined dataset: {len(df):,} rows, {len(df.columns)} columns")

## 2. Basic Statistics

In [ ]:
def distribution_summary(df: pd.DataFrame, col: str) -> pd.Series:
    """Return key descriptive stats for a numeric column."""
    return pd.Series({
        'count': df[col].count(),
        'mean': df[col].mean(),
        'median': df[col].median(),
        'std': df[col].std(),
        'skew': df[col].skew(),
        'kurtosis': df[col].kurtosis(),
        'min': df[col].min(),
        'max': df[col].max()
    })


numeric_cols = ['trip_distance', 'fare_amount', 'total_amount',
                'trip_duration_min', 'speed_mph']

stats_table = pd.DataFrame({col: distribution_summary(df, col)
                            for col in numeric_cols if col in df.columns}).T
display(stats_table.round(2))

## 3. Hourly Demand Pattern

In [ ]:
def plot_hourly_demand(df: pd.DataFrame):
    """24-hour demand line chart."""
    hourly = df.groupby('pickup_hour').size().reset_index(name='trip_count')

    fig, ax = plt.subplots(figsize=(14, 6))
    ax.fill_between(hourly['pickup_hour'], hourly['trip_count'],
                    alpha=0.3, color='#2196F3')
    ax.plot(hourly['pickup_hour'], hourly['trip_count'],
            color='#1565C0', linewidth=2.5, marker='o', markersize=6)

    ax.set_xlabel('Hour of Day', fontsize=13)
    ax.set_ylabel('Number of Trips', fontsize=13)
    ax.set_title('🕐 Hourly Taxi Demand Pattern', fontsize=16, fontweight='bold')
    ax.set_xticks(range(24))
    ax.set_xticklabels([f'{h:02d}:00' for h in range(24)], rotation=45)

    # Annotate peak
    peak_hour = hourly.loc[hourly['trip_count'].idxmax()]
    ax.annotate(f"Peak: {peak_hour['pickup_hour']:.0f}:00\n{peak_hour['trip_count']:,.0f} trips",
                xy=(peak_hour['pickup_hour'], peak_hour['trip_count']),
                xytext=(peak_hour['pickup_hour'] + 2, peak_hour['trip_count'] * 1.05),
                arrowprops=dict(arrowstyle='->', color='red'),
                fontsize=11, color='red', fontweight='bold')

    plt.tight_layout()
    save_figure(fig, 'hourly_demand')
    return fig


plot_hourly_demand(df)

## 4. Weekly Demand Heatmap

In [ ]:
def plot_weekly_heatmap(df: pd.DataFrame):
    """Hour × Day-of-Week demand intensity heatmap."""
    pivot = df.groupby(['pickup_day_of_week', 'pickup_hour']).size().unstack(fill_value=0)

    day_names = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']

    fig, ax = plt.subplots(figsize=(16, 6))
    sns.heatmap(pivot, cmap='YlOrRd', ax=ax, linewidths=0.5,
                xticklabels=[f'{h:02d}' for h in range(24)],
                yticklabels=day_names, fmt=',')

    ax.set_xlabel('Hour of Day', fontsize=13)
    ax.set_ylabel('Day of Week', fontsize=13)
    ax.set_title('📅 Demand Heatmap: Day of Week × Hour', fontsize=16, fontweight='bold')

    plt.tight_layout()
    save_figure(fig, 'weekly_heatmap')
    return fig


plot_weekly_heatmap(df)

## 5. Top 10 Pickup Zones

In [ ]:
def plot_top_zones(df: pd.DataFrame, n: int = 10):
    """Bar chart of the top n pickup zones by trip count."""
    top = (df.groupby('PULocationID').size()
           .reset_index(name='trip_count')
           .sort_values('trip_count', ascending=False)
           .head(n))

    fig, ax = plt.subplots(figsize=(14, 6))
    bars = ax.barh(top['PULocationID'].astype(str), top['trip_count'],
                   color=sns.color_palette('viridis', n))

    ax.set_xlabel('Number of Trips', fontsize=13)
    ax.set_ylabel('Pickup Zone ID', fontsize=13)
    ax.set_title(f'📍 Top {n} Pickup Zones by Trip Volume', fontsize=16, fontweight='bold')
    ax.invert_yaxis()

    # Add value labels
    for bar, count in zip(bars, top['trip_count']):
        ax.text(bar.get_width() + bar.get_width() * 0.01,
                bar.get_y() + bar.get_height() / 2,
                f'{count:,}', va='center', fontsize=10)

    plt.tight_layout()
    save_figure(fig, 'top_zones')
    return fig


plot_top_zones(df)

## 6. Trip Distance Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Histogram + KDE
sns.histplot(df['trip_distance'], kde=True, bins=60, ax=axes[0],
             color='#42A5F5', edgecolor='white')
axes[0].set_title('Trip Distance Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Distance (miles)')
axes[0].set_xlim(0, df['trip_distance'].quantile(0.99))

# Box plot
sns.boxplot(x=df['trip_distance'], ax=axes[1], color='#66BB6A')
axes[1].set_title('Trip Distance Box Plot', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Distance (miles)')
axes[1].set_xlim(0, df['trip_distance'].quantile(0.99))

plt.tight_layout()
save_figure(fig, 'trip_distance_distribution')

## 7. Monthly Trend

In [ ]:
monthly = df.groupby('pickup_month').size().reset_index(name='trip_count')

fig, ax = plt.subplots(figsize=(12, 6))
ax.bar(monthly['pickup_month'], monthly['trip_count'],
       color=sns.color_palette('coolwarm', len(monthly)), edgecolor='white')

ax.set_xlabel('Month', fontsize=13)
ax.set_ylabel('Total Trips', fontsize=13)
ax.set_title('📆 Monthly Trip Volume', fontsize=16, fontweight='bold')
ax.set_xticks(monthly['pickup_month'])
ax.set_xticklabels(['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
                     'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'][:len(monthly)])

for i, row in monthly.iterrows():
    ax.text(row['pickup_month'], row['trip_count'] + row['trip_count'] * 0.01,
            f"{row['trip_count']:,}", ha='center', fontsize=10)

plt.tight_layout()
save_figure(fig, 'monthly_trend')

## 8. Fare Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

sns.histplot(df['fare_amount'], kde=True, bins=80, ax=ax,
             color='#FFA726', edgecolor='white')

ax.set_title('💰 Fare Amount Distribution', fontsize=16, fontweight='bold')
ax.set_xlabel('Fare Amount ($)', fontsize=13)
ax.set_xlim(0, df['fare_amount'].quantile(0.99))

# Add median line
median_fare = df['fare_amount'].median()
ax.axvline(median_fare, color='red', linestyle='--', linewidth=2,
           label=f'Median: ${median_fare:.2f}')
ax.legend(fontsize=12)

plt.tight_layout()
save_figure(fig, 'fare_distribution')

## 9. Yellow vs Green Taxi Comparison

In [ ]:
if 'taxi_type' in df.columns:
    type_counts = df['taxi_type'].value_counts()

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Pie chart
    colors = ['#FDD835', '#66BB6A']
    axes[0].pie(type_counts, labels=type_counts.index,
                autopct='%1.1f%%', colors=colors, startangle=90,
                textprops={'fontsize': 13})
    axes[0].set_title('Trip Share by Taxi Type', fontsize=14, fontweight='bold')

    # Hourly comparison
    for t_type, color in zip(['yellow', 'green'], colors):
        subset = df[df['taxi_type'] == t_type]
        hourly = subset.groupby('pickup_hour').size()
        axes[1].plot(hourly.index, hourly.values, label=t_type.title(),
                     color=color, linewidth=2.5)

    axes[1].set_title('Hourly Demand by Taxi Type', fontsize=14, fontweight='bold')
    axes[1].set_xlabel('Hour')
    axes[1].set_ylabel('Trips')
    axes[1].legend()

    plt.tight_layout()
    save_figure(fig, 'yellow_vs_green')

---
**Next →** `04_statistical_analysis.ipynb`